In [ ]:
from pathlib import Path
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text


DB_USER = "postgres"
DB_PASSWORD = "admin123"       
DB_HOST = "127.0.0.1"          
DB_PORT = "5432"                
DB_NAME = "shopmetricsdb"         

SAMPLE_SESSIONS = 150000
RANDOM_SEED = 42


engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
with engine.connect() as conn:
    connected_db = conn.execute(text("SELECT current_database();")).scalar()
    print(f"1/6 Connected to PostgreSQL database: '{connected_db}'")


search_dirs = [
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "Desktop",
    Path.home() / "Documents",
]

csv_path = r"C:\project2\2019-Dec.csv"
for folder in search_dirs:
    for fname in ["2019-Dec.csv", "2019-Dec.csv.zip"]:
        candidate = folder / fname
        if candidate.exists():
            csv_path = str(candidate)
            break
    if csv_path:
        break

if not csv_path:
    raise FileNotFoundError(
        "Place '2019-Dec.csv' in your Downloads folder or current notebook folder and re-run."
    )

print(f"2/6 Reading dataset from: {csv_path}")
use_cols = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session"
]
df = pd.read_csv(csv_path, usecols=use_cols)

df = df.dropna(subset=["user_session", "user_id", "product_id"]).copy()
df = df[df["price"] > 0].copy()
df["event_time"] = pd.to_datetime(df["event_time"].str.replace(" UTC", "", regex=False))

rng = np.random.default_rng(RANDOM_SEED)
unique_sessions = df["user_session"].unique()
if len(unique_sessions) > SAMPLE_SESSIONS:
    selected_sessions = rng.choice(unique_sessions, size=SAMPLE_SESSIONS, replace=False)
    df = df[df["user_session"].isin(selected_sessions)].copy()

df = df.sort_values(["user_session", "event_time"]).reset_index(drop=True)
print(f"    Loaded {len(df):,} events across {df['user_session'].nunique():,} sessions.")


print("3/6 Building dim_products...")
dim_products = (
    df.groupby("product_id", as_index=False)
    .agg(
        category_id=("category_id", "first"),
        category_code=("category_code", "first"),
        brand=("brand", "first"),
        price=("price", "median"),
    )
)

cosmetic_categories = [
    "Skincare Serum", "Matte Lipstick", "Hydrating Cream", "Gel Nail Polish",
    "Volume Mascara", "Hair Repair Oil", "Facial Cleanser", "Setting Powder"
]
dim_products["brand"] = dim_products["brand"].fillna("Lumina Beauty").str.title()
dim_products["category"] = dim_products["category_code"].fillna(
    pd.Series(rng.choice(cosmetic_categories, size=len(dim_products)), index=dim_products.index)
)
dim_products["category"] = (
    dim_products["category"].str.split(".").str[-1].str.replace("_", " ").str.title()
)
dim_products["product_name"] = (
    dim_products["brand"] + " " + dim_products["category"] + " #" + dim_products["product_id"].astype(str).str[-4:]
)
dim_products = dim_products[["product_id", "product_name", "category", "brand", "price"]]


print("4/6 Building dim_session_attribution and dim_users...")
# Using 'min' for user_id matches SQL MIN(user_id) in fct_sessions 100%
sessions_df = (
    df.groupby("user_session", as_index=False)
    .agg(user_id=("user_id", "min"), session_start=("event_time", "min"))
    .rename(columns={"user_session": "session_id"})
)

n_sess = len(sessions_df)
sessions_df["traffic_source"] = rng.choice(
    ["Organic Search", "Paid Social", "Direct", "Email Campaign", "Referral"],
    size=n_sess,
    p=[0.34, 0.26, 0.20, 0.12, 0.08]
)
sessions_df["device_type"] = rng.choice(
    ["Mobile", "Desktop", "Tablet"],
    size=n_sess,
    p=[0.58, 0.35, 0.07]
)
sessions_df["country"] = rng.choice(
    ["United States", "United Kingdom", "India", "Germany", "Canada", "Australia"],
    size=n_sess,
    p=[0.36, 0.18, 0.16, 0.12, 0.10, 0.08]
)

dim_session_attribution = sessions_df[["session_id", "traffic_source", "device_type", "country"]]

dim_users = (
    sessions_df.sort_values("session_start")
    .groupby("user_id", as_index=False)
    .agg(
        first_seen=("session_start", "min"),
        country=("country", "first"),
        primary_device=("device_type", "first"),
        acquisition_source=("traffic_source", "first"),
    )
)


print("5/6 Writing tables to PostgreSQL...")
raw_events = df[["event_time", "event_type", "product_id", "user_id", "user_session", "price"]].rename(
    columns={"user_session": "session_id"}
)

raw_events.to_sql("raw_events", engine, if_exists="replace", index=False, chunksize=50000, method="multi")
dim_products.to_sql("dim_products", engine, if_exists="replace", index=False, chunksize=25000, method="multi")
dim_users.to_sql("dim_users", engine, if_exists="replace", index=False, chunksize=25000, method="multi")
dim_session_attribution.to_sql("dim_session_attribution", engine, if_exists="replace", index=False, chunksize=25000, method="multi")

print("6/6 ETL Complete! Loaded: raw_events, dim_products, dim_users, dim_session_attribution.")

1/6 Connected to PostgreSQL database: 'shopmetricsdb'
2/6 Reading dataset from: C:\project2\2019-Dec.csv
    Loaded 630,220 events across 150,000 sessions.
3/6 Building dim_products...
4/6 Building dim_session_attribution and dim_users...
5/6 Writing tables to PostgreSQL...
6/6 ETL Complete! Loaded: raw_events, dim_products, dim_users, dim_session_attribution.


In [ ]:
from collections import Counter
from itertools import combinations
from urllib.parse import quote_plus
import pandas as pd
from sqlalchemy import create_engine


DB_USER = "postgres"
DB_PASSWORD = "admin123"
DB_HOST = "127.0.0.1"
DB_PORT = "5432"                
DB_NAME = "shopmetricsdbp"        

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("1/3 Loading purchased baskets from PostgreSQL...")
purchases = pd.read_sql(
    """
    SELECT DISTINCT r.session_id, r.product_id, p.product_name, p.price
    FROM raw_events r
    JOIN dim_products p ON r.product_id = p.product_id
    WHERE r.event_type = 'purchase'
    """,
    engine,
)

total_orders = purchases["session_id"].nunique()
item_counts = purchases.groupby("product_id")["session_id"].nunique().to_dict()
product_meta = (
    purchases.drop_duplicates("product_id")
    .set_index("product_id")[["product_name", "price"]]
    .to_dict("index")
)

baskets = purchases.groupby("session_id")["product_id"].apply(lambda s: sorted(set(s)))
multi_baskets = baskets[baskets.str.len() >= 2]
print(f"    Analyzed {total_orders:,} orders ({len(multi_baskets):,} multi-item baskets).")


print("2/3 Mining product affinity rules...")
pair_counts = Counter()
for items in multi_baskets:
    for a, b in combinations(items, 2):
        pair_counts[(a, b)] += 1
        pair_counts[(b, a)] += 1

rules = []
for (ante_id, cons_id), co_count in pair_counts.items():
    if co_count < 3:
        continue
    support = co_count / total_orders
    confidence = co_count / item_counts[ante_id]
    consequent_support = item_counts[cons_id] / total_orders
    lift = confidence / consequent_support

    ante_info = product_meta[ante_id]
    cons_info = product_meta[cons_id]
    bundle_price = round((ante_info["price"] + cons_info["price"]) * 0.90, 2)

    if lift >= 20 and confidence >= 0.25:
        action = "Cart Drawer 1-Click Cross-Sell (10% Bundle)"
    elif lift >= 10:
        action = "PDP 'Frequently Bought Together' Widget"
    else:
        action = "Post-Purchase Email Upsell Offer"

    rules.append({
        "antecedent_id": ante_id,
        "antecedent_product": ante_info["product_name"],
        "consequent_id": cons_id,
        "consequent_product": cons_info["product_name"],
        "pair_count": co_count,
        "support_pct": round(support * 100, 3),
        "confidence_pct": round(confidence * 100, 2),
        "lift": round(lift, 2),
        "bundle_price_10pct_off": bundle_price,
        "recommendation": action,
    })

mart_basket_rules = (
    pd.DataFrame(rules)
    .sort_values(["lift", "confidence_pct"], ascending=[False, False])
    .head(50)
    .reset_index(drop=True)
)
mart_basket_rules.insert(0, "rule_id", mart_basket_rules.index + 1)


print("3/3 Saving mart_basket_rules and exporting basket_recommendations.csv...")
mart_basket_rules.to_sql("mart_basket_rules", engine, if_exists="replace", index=False)
mart_basket_rules.to_csv("basket_recommendations.csv", index=False)

print(f"Done! Created {len(mart_basket_rules)} rules and saved 'basket_recommendations.csv'.")

1/3 Loading purchased baskets from PostgreSQL...
    Analyzed 5,094 orders (4,493 multi-item baskets).
2/3 Mining product affinity rules...
3/3 Saving mart_basket_rules and exporting basket_recommendations.csv...
Done! Created 50 rules and saved 'basket_recommendations.csv'.
